# Reward Model with Bradley-Terry Loss

**Goal**: After this session, you will be able to implement a reward model that scores (prompt, completion) pairs and train it with pairwise preference data.

**Time**: 30 minutes.

## What and Why
A reward model is the bridge between human preferences and RL training. It takes a (prompt, completion) pair and outputs a scalar score. During RLHF, this score is the reward signal that PPO optimizes against. If the reward model is wrong, everything downstream is wrong — reward hacking, mode collapse, and misalignment all trace back here.

## Key Formula
The Bradley-Terry model defines the probability that completion $y_w$ is preferred over $y_l$ as:

$$P(y_w \succ y_l | x) = \sigma(r_\theta(x, y_w) - r_\theta(x, y_l))$$

The loss maximizes the log-likelihood of observed preferences:

$$\mathcal{L} = -\mathbb{E}\left[\log \sigma(r_\theta(x, y_w) - r_\theta(x, y_l))\right]$$

## References
- Ouyang et al., "Training language models to follow instructions with human feedback" (InstructGPT, 2022)
- Kimi K2 Technical Report (Moonshot AI, 2025) — current open-weight SOTA, 1T params MoE
- Bradley & Terry, "Rank Analysis of Incomplete Block Designs" (1952)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from jaxtyping import Float
from torch import Tensor
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_ID = "moonshotai/Kimi-K2-Base"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model.eval()

text = "hello world"
inputs = tokenizer(text, return_tensors="pt")

outs = model(**inputs, output_hidden_states=True, use_cache=False)


config.json: 0.00B [00:00, ?B/s]

configuration_deepseek.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/moonshotai/Kimi-K2-Base:
- configuration_deepseek.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
You are using a model of type kimi_k2 to instantiate a model of type deepseek_v3. This is not supported for all configurations of models and can yield errors.


modeling_deepseek.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/moonshotai/Kimi-K2-Base:
- modeling_deepseek.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


ImportError: cannot import name 'is_torch_fx_available' from 'transformers.utils.import_utils' (/share/datasets/home/lofty/TorchLeet/.venv/lib/python3.12/site-packages/transformers/utils/import_utils.py)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from jaxtyping import Float
from torch import Tensor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Synthetic preference dataset ---
# In a real RLHF pipeline, you'd run a frozen base LM on (prompt, completion)
# pairs and extract the final-layer hidden states: shape (batch, seq_len, hidden_dim).
# Here we generate synthetic hidden states directly — same shape, same interface,
# no 61-layer forward pass required.

torch.manual_seed(42)

HIDDEN_DIM = 256   # scaled down from 7168 (Kimi K2)
SEQ_LEN = 64       # scaled down from 128K context
N_PAIRS = 1024     # typical RM datasets are 50K-500K pairs

def generate_hidden_states() -> Float[Tensor, "seq hidden"]:
    """Simulate a base LM's final-layer output for one completion."""
    return torch.randn(SEQ_LEN, HIDDEN_DIM, device=device)

def oracle_reward(hidden_states: Float[Tensor, "seq hidden"]) -> float:
    """Ground-truth reward: mean of last token's hidden state.
    The reward model must learn to approximate this."""
    return hidden_states[-1].mean().item()

def make_preference_pair():
    """Generate a (chosen, rejected) pair ordered by oracle reward."""
    hidden_a = generate_hidden_states()
    hidden_b = generate_hidden_states()

    if oracle_reward(hidden_a) > oracle_reward(hidden_b):
        return hidden_a, hidden_b
    else:
        return hidden_b, hidden_a

# Build dataset: pairs of (chosen_hidden_states, rejected_hidden_states)
chosen_list, rejected_list = [], []
for _ in range(N_PAIRS):
    c, r = make_preference_pair()
    chosen_list.append(c)
    rejected_list.append(r)

chosen_hidden = torch.stack(chosen_list)     # (N, seq_len, hidden_dim)
rejected_hidden = torch.stack(rejected_list) # (N, seq_len, hidden_dim)

print(f"Dataset: {N_PAIRS} preference pairs")
print(f"Hidden states shape: {chosen_hidden.shape}")
print(f"  -> same shape as base_model(tokens).hidden_states[-1]")
print(f"  -> to use a real model instead: Qwen/Qwen3-0.6B (0.6B, GQA, SwiGLU, RoPE)")
print(f"Device: {device}")

## Part 1: Reward Model Architecture
Build a reward model that takes a sequence of hidden states and outputs a scalar reward. The key design choice: use **last-token pooling** — the hidden state at the final position becomes the sequence representation, then project to a scalar.

**Predict before you code**: What shape is the input? What shape is the output? Why use the last token specifically (think about how autoregressive LMs work)?

<details><summary>Hint: pooling strategy</summary>In a real reward model built on top of a causal LM, the last token has attended to all previous tokens, so it carries the most complete representation of the full sequence.</details>

In [ ]:
class RewardModel(nn.Module):
    """Reward model: hidden states -> scalar reward via last-token pooling."""

    def __init__(self, hidden_dim: int):
        super().__init__()
        # TODO: Define a reward head that projects hidden_dim -> 1
        pass

    def forward(
        self, hidden_states: Float[Tensor, "batch seq hidden"]
    ) -> Float[Tensor, "batch"]:
        # TODO: Extract last token's hidden state, project to scalar, squeeze
        pass

In [ ]:
# --- Part 1 Validation ---
torch.manual_seed(42)
rm_test = RewardModel(HIDDEN_DIM).to(device)
test_input = torch.randn(4, SEQ_LEN, HIDDEN_DIM, device=device)

result = rm_test(test_input)

# Shape
assert result.shape == (4,), f"Shape: expected (4,), got {result.shape}"
print(f"  Shape: {result.shape} -- correct")

# Diagnostics
print(f"  Range: [{result.min():.4f}, {result.max():.4f}]")
print(f"  Mean:  {result.mean():.4f}, Std: {result.std():.4f}")

# Property: output should be a scalar per sample (not all identical)
assert not torch.all(result == result[0]), "All outputs identical — reward head may not be using the input"
print(f"  Non-constant outputs -- correct")

# Gradient flows
result.sum().backward()
assert rm_test.reward_head.weight.grad is not None, "No gradients flowing to reward head"
print(f"  Gradient flow -- correct")

print("\nPart 1 complete.")

## Part 2: Bradley-Terry Pairwise Loss
Implement the loss function that trains the reward model from preference pairs. Given rewards for a chosen and rejected completion, the loss is the negative log-likelihood under the Bradley-Terry model.

**Predict before you code**: If the model correctly assigns higher reward to chosen (+5) vs rejected (-5), what should the loss be close to? What if both rewards are equal?

In [ ]:
def bradley_terry_loss(
    rewards_chosen: Float[Tensor, "batch"],
    rewards_rejected: Float[Tensor, "batch"],
) -> Float[Tensor, ""]:
    """Compute Bradley-Terry pairwise ranking loss. Returns scalar mean loss."""
    # TODO: Implement the BT loss: -mean(log(sigmoid(r_chosen - r_rejected)))
    pass

In [ ]:
# --- Part 2 Validation ---
torch.manual_seed(42)

# Test 1: Perfect separation — loss should be near 0
r_chosen = torch.tensor([5.0, 3.0, 10.0])
r_rejected = torch.tensor([-5.0, -3.0, -10.0])
loss_perfect = bradley_terry_loss(r_chosen, r_rejected)
print(f"  Perfect separation loss: {loss_perfect:.6f}")
assert loss_perfect < 0.01, f"Loss should be near 0 for perfect separation, got {loss_perfect:.4f}"
print(f"  Perfect separation -- correct")

# Test 2: Equal rewards — loss should be log(2) ≈ 0.693
r_equal = torch.tensor([0.0, 0.0, 0.0])
loss_equal = bradley_terry_loss(r_equal, r_equal)
print(f"  Equal rewards loss: {loss_equal:.6f}")
try:
    assert torch.allclose(loss_equal, torch.tensor(0.6931), atol=1e-3), f"Expected ~0.693, got {loss_equal:.4f}"
except AssertionError:
    print(f"  YOUR output:  {loss_equal.item():.6f}")
    print(f"  EXPECTED:     0.693147")
    print(f"  DIFFERENCE:   {abs(loss_equal.item() - 0.693147):.6f}")
    raise
print(f"  Equal rewards (log(2)) -- correct")

# Test 3: Wrong ordering — loss should be > log(2)
loss_wrong = bradley_terry_loss(r_rejected, r_chosen)
print(f"  Wrong ordering loss: {loss_wrong:.6f}")
assert loss_wrong > 0.693, f"Loss should be > log(2) for wrong ordering, got {loss_wrong:.4f}"
print(f"  Wrong ordering penalty -- correct")

# Test 4: Scalar output
assert loss_perfect.shape == (), f"Loss should be scalar, got shape {loss_perfect.shape}"
print(f"  Scalar output -- correct")

print("\nPart 2 complete.")

## Part 3: Training Loop
Train the reward model on the preference dataset. Iterate over minibatches of (chosen, rejected) pairs, forward both through the reward model, compute BT loss, and update.

**Predict before you code**: What should the training accuracy (fraction where r_chosen > r_rejected) converge to? What would it mean if accuracy plateaus at 50%?

In [ ]:
def train_reward_model(
    model: RewardModel,
    chosen: Float[Tensor, "n seq hidden"],
    rejected: Float[Tensor, "n seq hidden"],
    n_epochs: int = 3,
    batch_size: int = 32,
    lr: float = 1e-3,
) -> list[dict]:
    """Train reward model on preference pairs. Returns list of per-epoch stats."""
    # TODO: Set up optimizer
    # TODO: Training loop:
    #   - Shuffle indices each epoch
    #   - Iterate minibatches
    #   - Forward chosen and rejected through model
    #   - Compute BT loss
    #   - Compute accuracy (fraction where r_chosen > r_rejected)
    #   - Backprop and step
    #   - Track epoch loss and accuracy
    # Return: [{"epoch": i, "loss": float, "accuracy": float}, ...]
    pass

In [ ]:
# --- Part 3 Validation ---
torch.manual_seed(42)
reward_model = RewardModel(HIDDEN_DIM).to(device)

stats = train_reward_model(reward_model, chosen_hidden, rejected_hidden, n_epochs=5)

# Stats structure
assert isinstance(stats, list), f"Expected list, got {type(stats)}"
assert len(stats) == 5, f"Expected 5 epoch stats, got {len(stats)}"
assert all("loss" in s and "accuracy" in s for s in stats), "Each stat dict needs 'loss' and 'accuracy' keys"
print(f"  Stats structure -- correct")

# Loss should decrease
print("  Epoch losses:", [round(s["loss"], 4) for s in stats])
assert stats[-1]["loss"] < stats[0]["loss"], f"Loss should decrease: {stats[0]['loss']:.4f} -> {stats[-1]['loss']:.4f}"
print(f"  Loss decreasing -- correct")

# Accuracy should improve
print("  Epoch accs:  ", [round(s["accuracy"], 3) for s in stats])
assert stats[-1]["accuracy"] > 0.60, f"Final accuracy should be > 60%, got {stats[-1]['accuracy']:.1%}"
print(f"  Accuracy improving -- correct")

# Sanity: accuracy is between 0 and 1
assert all(0 <= s["accuracy"] <= 1 for s in stats), "Accuracy should be in [0, 1]"
print(f"  Accuracy range -- correct")

print(f"\nPart 3 complete.")

## Part 4: Reward Distribution Analysis
After training, examine the reward model's behavior. Compute rewards for all chosen and rejected completions and compare the distributions. A well-trained RM should show clear separation.

**Predict before you code**: Should the reward distributions overlap? What would perfect separation look like? What would overfitting look like?

In [ ]:
def evaluate_reward_model(
    model: RewardModel,
    chosen: Float[Tensor, "n seq hidden"],
    rejected: Float[Tensor, "n seq hidden"],
) -> dict:
    """Evaluate reward model. Returns dict with rewards and summary statistics."""
    # TODO: Compute rewards for all chosen and rejected (no grad)
    # TODO: Compute:
    #   - accuracy: fraction where r_chosen > r_rejected
    #   - mean_chosen, mean_rejected: average rewards
    #   - mean_margin: average (r_chosen - r_rejected)
    #   - rewards_chosen, rewards_rejected: the raw reward tensors (for plotting)
    # Return as a dict with these keys
    pass

In [ ]:
# --- Part 4 Validation ---
eval_results = evaluate_reward_model(reward_model, chosen_hidden, rejected_hidden)

# Required keys
required_keys = ["accuracy", "mean_chosen", "mean_rejected", "mean_margin", "rewards_chosen", "rewards_rejected"]
for key in required_keys:
    assert key in eval_results, f"Missing key: {key}"
print(f"  All keys present -- correct")

# Separation: mean chosen > mean rejected
print(f"  Mean chosen:   {eval_results['mean_chosen']:.4f}")
print(f"  Mean rejected: {eval_results['mean_rejected']:.4f}")
print(f"  Mean margin:   {eval_results['mean_margin']:.4f}")
assert eval_results["mean_chosen"] > eval_results["mean_rejected"], "Chosen rewards should be higher than rejected"
print(f"  Reward separation -- correct")

# Positive margin
assert eval_results["mean_margin"] > 0, f"Mean margin should be positive, got {eval_results['mean_margin']:.4f}"
print(f"  Positive margin -- correct")

# Accuracy matches Part 3
print(f"  Eval accuracy: {eval_results['accuracy']:.3f}")
assert eval_results["accuracy"] > 0.60, f"Accuracy too low: {eval_results['accuracy']:.1%}"
print(f"  Accuracy threshold -- correct")

# Tensor shapes
assert eval_results["rewards_chosen"].shape == (N_PAIRS,), f"Wrong chosen shape: {eval_results['rewards_chosen'].shape}"
assert eval_results["rewards_rejected"].shape == (N_PAIRS,), f"Wrong rejected shape: {eval_results['rewards_rejected'].shape}"
print(f"  Tensor shapes -- correct")

print(f"\nPart 4 complete.")
print(f"\n{'='*50}")
print(f"All parts complete. Final stats:")
print(f"  Accuracy: {eval_results['accuracy']:.1%}")
print(f"  Mean margin: {eval_results['mean_margin']:.4f}")

## Session Debrief

Without scrolling up, answer in your head:
1. What is the Bradley-Terry loss formula? Why sigmoid of the reward *difference*?
2. Why does a reward model use last-token pooling instead of mean pooling?
3. If training accuracy is 100% but validation accuracy is 55%, what happened?

**Challenge**: Close this notebook, open a blank one, and rewrite Part 2 (the BT loss) from scratch without looking back.

## Extension (Optional)
Try one of these variations:
- **Margin-based BT loss**: Add a margin term: $-\log \sigma(r_w - r_l - m)$ where $m > 0$. When would this help?
- **What breaks**: Set learning rate to 1.0 and retrain. What happens to the reward scale? Why does this matter for downstream PPO?
- **Mean pooling variant**: Replace last-token pooling with mean pooling. Does accuracy change? Why or why not for this synthetic setup?